In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from collections import defaultdict, Counter
import pprint

In [5]:
# Загрузка данных
def load_data(file_name: str) -> list[list[int]]:
    """
    Загружает данные из JSON-файла, где каждая строка является отдельным JSON-объектом.

    Функция построчно читает файл, парсит каждую непустую строку как JSON
    и добавляет полученный объект в список.

    Args
    ----------
        file_name (str): Путь к JSON-файлу, где каждая строка содержит JSON-массив целых чисел.

    Returns
    ----------
        list[list[int]]: Список сессий, где каждая сессия представлена списком целых чисел.
    """
    sessions = []
    with open(file_name) as file:
        for line in file:
            line = line.strip()
            if line:
                sessions.append(json.loads(line))
    return sessions

In [2]:
def train_test_split(sessions: list[list[int]]) -> tuple[list[list[int]], list[int]]:
    """
    Разбиение сессий на train и test.

    Для каждой сессии:
      - все товары кроме последнего становятся
        обучающей сессией
      - последний товар становится тестовой целью

    Возвращаемые списки выровнены по индексу:
    test_targets[i] — товар, который нужно
    предсказать по train_sessions[i] как истории.

    Args
    ----------
    sessions : list of lists of ints
        Каждый вложенный список — одна сессия ID
        товаров, упорядоченная по времени.
        Все сессии содержат не менее 3 товаров.

    Returns
    -------
    train_sessions : list of lists of ints
        Сессии для обучения (исходные без последнего товара).
    test_targets : list of ints
        Следующий товар для предсказания по каждой сессии.
    """
    train_sessions = [session[:-1] for session in sessions]
    test_targets = [session[-1] for session in sessions]
    return train_sessions, test_targets

In [14]:
def build_transition_graph(train_sessions: list[list[int]]) -> dict[int, dict[int, float]]:
    """
    Построение графа переходов с вероятностями P(j | i).

    Для каждой обучающей сессии считаются все переходы между
    соседними товарами из i в j. По собранным связям
    вычисляются условные вероятности:
        P(j | i) = (количество случаев i перешло в j) / (сумма всех переходов из i в любой товар, в том числе j)

    Вершины графа - ID товаров. Ребра направленные, каждое ребро из i в j имеет вес - P(j | i).

    Args
    ----------
    train_sessions
        Список обучающих сессий.

    Returns
    -------
    dict[int, dict[int, float]]
        Словарь, где ключ - это текущий товар i,
        значение - это словарь {j: P(j | i)}.
        Если из товара i нет переходов, он отсутствует
        в возвращаемом словаре.

    dict[int, Counter[int, int]]
        Словарь всех переходов и их количеств для проверки того,
        что алгоритм работает верно. 
    """    
    # Создаю словарь с ключом - ID и словарем, который автоматически считает количество связей.
    transitions_btw_items = defaultdict(Counter)

    # Перебор всех сессий.
    for session in train_sessions:
        # Перебор всех предметов.
        for item_idx in range(len(session) - 1):
            current_item = session[item_idx]
            next_item = session[item_idx + 1]
            
            transitions_btw_items[current_item][next_item] += 1
    
    graph_probabilities = dict()
    for current_item, next_items_counts in transitions_btw_items.items():
        total_transitions = sum(next_items_counts.values())
        graph_probabilities[current_item] = {j_item: transitions_count / total_transitions 
                                                for j_item, transitions_count in next_items_counts.items()}
    
    return graph_probabilities, transitions_btw_items

In [55]:
def get_most_popular_items(train_sessions: list[list[int]]) -> list[int]:
    """
    Возвращает список товаров, отсортированных по убыванию популярности.
    
    Args
    ----------
    train_sessions
        Список обучающих сессий

    Returns
    -------
    list[int]
        Список товаров, отсортированных по частоте встречаемости.
    """
    # Собираю все товары из всех сессий
    all_items = []
    for session in train_sessions:
        all_items.extend(session)
    
    # Считаю частоту каждого товара
    popularity = Counter(all_items)
    
    # Сортирую по убыванию частоты
    sorted_items = [item for item, _ in popularity.most_common()]
    
    return sorted_items[:10]

In [ ]:
def recommend_next(graph_probabilities: dict[int, dict[int, float]], 
                   last_item: int, most_popular_items: list[int]) -> list[int]:
    """
    Рекомендация следующих товаров на основе графа переходов.
    
    Если для last_item есть переходы - возвращает топ-10 самых вероятных.
    Если нет — возвращаем топ-10 популярных товаров.
    
    Args
    ----------
    graph_probabilities
        Граф вероятностей переходов из build_transition_graph
    last_item
        Последнего товар в сессии
    most_popular_items
        Список товаров, отсортированных по популярности
        
    Returns
    -------
    list[int]
        Список из 10 рекомендуемых товаров
    """
    k = 10
    recommendations = []
    
    # Проверяю - есть ли переходы из последнего товара?
    if last_item_idx in graph_probabilities:
        # Сортирую по убыванию вероятности
        sorted_probabilities = sorted(
            graph_probabilities[last_item].items(),
            key=lambda x: -x[1]
        )
        
        recommendations = [item for item, _ in sorted_probabilities[:k]]
    
    # Если рекомендаций меньше k. Добавляю популярные товары
    if len(recommendations) < k:
        for item in most_popular_items:
            if item != last_item and item not in recommendations:
                recommendations.append(item)
            if len(recommendations) == k:
                break
    
    return recommendations[:k]

In [72]:
def hit_at_k(recommendations: list[list[int]], true_items: list[int], k: int = 10) -> float:
    """
    Вычисление Hit@K для списка предсказаний.

    Parameters
    ----------
    recommendations : list of lists of ints
        recommendations[i] — ранжированный список
        рекомендаций для i-го примера.
    true_items : list of ints
        true_items[i] — истинный следующий товар
        для i-го примера.
    k : int
        Отсечка top-K (по умолчанию 10).

    Returns
    -------
    float
        Hit@K, значение от 0 до 1.
    """
    assert len(recommendations) == len(true_items), \
        "recommendations и true_items должны совпадать по длине"

    hits = 0
    for recs, true_item in zip(recommendations, true_items):
        if true_item in recs[:k]:
            hits += 1

    return hits / len(true_items)

In [6]:
# Загрузка сессий
sessions = load_data("sessions.jsonl")

In [74]:
# Формирование обучающей и тестовой выборок.
train_sessions, test_targets = train_test_split(sessions)

In [15]:
# Граф вероятностей и получаю переходы для отладк.
graph_probabilities, transitions = build_transition_graph(train_sessions)

In [29]:
# Отладка. Проверяю, совпадает ли информация, полученная из EDA раздела с данной.
# У предмета 54 самая частая связь - 53, все верно
# Общее количество связей предмета 54 - около 2500, верно.
# В EDA был проведен анализ с учетом тестовой выборки и переходов в нее, поэтому здесь они не учитывались -
# число связей из 54 в другие товары меньше
print(sum(transitions[54].values()))
pprint.pprint(transitions[54])

2407
Counter({53: 407,
         329: 210,
         335: 171,
         114: 78,
         338: 70,
         260: 70,
         293: 59,
         376: 50,
         65: 45,
         212: 42,
         380: 34,
         257: 33,
         149: 24,
         394: 23,
         63: 23,
         247: 21,
         245: 18,
         153: 18,
         368: 16,
         226: 15,
         289: 14,
         266: 14,
         77: 13,
         233: 13,
         275: 13,
         205: 13,
         132: 13,
         86: 12,
         28: 12,
         362: 12,
         354: 12,
         144: 11,
         161: 11,
         281: 10,
         169: 10,
         277: 10,
         96: 9,
         390: 9,
         13: 9,
         92: 8,
         385: 8,
         320: 8,
         104: 8,
         262: 8,
         99: 8,
         373: 7,
         251: 7,
         328: 7,
         41: 7,
         194: 7,
         360: 7,
         97: 7,
         156: 6,
         333: 6,
         387: 6,
         191: 6,
         244: 6,

In [48]:
# Отладка
# Так же просматриваю получившиеся вероятности переходов для предмета 54.
max_item = max(graph_probabilities[54], key=graph_probabilities[54].get)
max_probability = graph_probabilities[54][max_item]

print(f"Макс. вероятность перехода - {max_probability}, товар - {max_item}") # Макс.вероятность - из 54 в 53.
pprint.pprint(graph_probabilities[54])

Макс. вероятность перехода - 0.16909015371832156, товар - 53
{0: 0.0004154549231408392,
 1: 0.0024927295388450354,
 2: 0.002077274615704196,
 3: 0.0008309098462816784,
 4: 0.002077274615704196,
 5: 0.0024927295388450354,
 6: 0.002077274615704196,
 7: 0.0004154549231408392,
 8: 0.0012463647694225177,
 12: 0.0012463647694225177,
 13: 0.003739094308267553,
 16: 0.0004154549231408392,
 17: 0.0004154549231408392,
 18: 0.0012463647694225177,
 19: 0.0004154549231408392,
 20: 0.0004154549231408392,
 21: 0.0008309098462816784,
 22: 0.0012463647694225177,
 23: 0.0012463647694225177,
 24: 0.0008309098462816784,
 25: 0.0004154549231408392,
 27: 0.0008309098462816784,
 28: 0.004985459077690071,
 29: 0.0008309098462816784,
 30: 0.0012463647694225177,
 31: 0.0004154549231408392,
 32: 0.002077274615704196,
 34: 0.0008309098462816784,
 35: 0.0008309098462816784,
 36: 0.0004154549231408392,
 37: 0.0004154549231408392,
 38: 0.0016618196925633569,
 39: 0.002077274615704196,
 40: 0.0004154549231408392,
 41

In [71]:
# Получаю самые популярные товары на случай, если будет мало связей у конкретного товара.
most_popular_items = get_most_popular_items(train_sessions)

In [78]:
# Делаю предсказания для каждой сессии
predictions = []
for session in train_sessions:
    last_item = session[-1]
    recomendations = recommend_next(graph_probabilities, last_item, most_popular_items)
    predictions.append(recomendations)

In [ ]:
# Вычисляю Hit@10.
hit_at_10 = hit_at_k(predictions, test_targets, k=10)

In [ ]:
# Для сравнения - бейзлайн - только популярные товары.
baseline_recommendations = [most_popular_items[:10] for _ in range(len(test_targets))]
baseline_hit = hit_at_k(baseline_recommendations, test_targets, k=10)

In [80]:
print("Финальное сравнение результатов:")
print(" ")

print(f"Hit@10 моей модели: {hit_at_10:.5f}")
print(" ")

print(f"Бейзлайн - только популярные - Hit@10: {baseline_hit_rate:.5f}")
print(" ")

difference = hit_at_10 - baseline_hit 
print(f"Разница качества моделей - {difference}")

Финальное сравнение результатов:
 
Hit@10 моей модели: 0.51423
 
Бейзлайн - только популярные - Hit@10: 0.38402
 
Разница качества моделей - 0.13021442495126706
